<a href="https://colab.research.google.com/github/imanuni/imanuni/blob/main/nom-verb-tf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.sparse import hstack, csr_matrix
import joblib


In [4]:
# Charger le fichier Excel

from google.colab import files

# Sélectionne ton fichier local
uploaded = files.upload()


Saving model.csv to model (1).csv


In [5]:
import os
print(os.listdir())  # Vérifie que model.xlsx est bien là


df = pd.read_csv("model.csv", encoding="utf-8")
# --- Détection automatique des colonnes utiles
titre_col = [c for c in df.columns if any(k in c.lower() for k in ["titre", "tokens", "title", "text", "headline"])][0]
label_col = [c for c in df.columns if any(k in c.lower() for k in ["label", "yes", "no", "is_", "decla", "décla", "target", "class", "classe"])][0]
df = df[[titre_col, label_col]].rename(columns={titre_col: "titre", label_col: "label"})

#  Affichage clair des 4 premières lignes
from IPython.display import display

print("=== Aperçu des 4 premières lignes ===")
display(df.head(4).style.set_properties(**{
    'text-align': 'right',      # Alignement à droite pour l'arabe
    'direction': 'rtl',         # Lecture de droite à gauche
    'font-size': '16px',        # Texte plus grand
    'font-family': 'Arial'      # Police claire pour l'arabe
}))



['.config', 'model.csv', 'model (1).csv', 'sample_data']
=== Aperçu des 4 premières lignes ===


,titre,label
0,"['عون:', 'لحكومه', 'تمثل', 'الجميع', 'مطالب', 'كتل', 'توخر', 'التاليف']",yes
1,"['زياره', 'ايرانيه', 'للراعي:', 'رساله', 'وتطمينات', 'وسجاده', 'بقطب', 'مخفيه']",no
2,"['تفاول', 'بحل', 'قضيه', 'الودايع:', 'اسس']",no
3,"['بوادر', 'حمله', 'عسكريه', 'اسراييليه', 'غربيه:', 'صنعاء', 'تتحضر', 'لتصعيد', 'وشيك']",no


In [ ]:

# --- Normalisation des labels
def normalize_label(x):
    if pd.isnull(x): return np.nan
    s = str(x).strip().lower()
    if s in ["yes", "1", "true", "vrai", "oui"]: return "declaration"
    if s in ["no", "0", "false", "non", "faux"]: return "non-declaration"
    if "yes" in s: return "declaration"
    return s

df["label"] = df["label"].apply(normalize_label)

print("=== Aperçu des 4 premières lignes ===")
display(df.head(4).style.set_properties(**{
    'text-align': 'right',      # Alignement à droite pour l'arabe
    'direction': 'rtl',         # Lecture de droite à gauche
    'font-size': '16px',        # Texte plus grand
    'font-family': 'Arial'      # Police claire pour l'arabe
}))


In [6]:
import pandas as pd
from IPython.display import display

# --- Liste des verbes déclaratifs (inchangée)
declarative_verbs = ["قال","أعلن","يدعو","أكد","أوضح","أشار","امر","ذكر","يدحض","شدد","نفى","تحدث","دعا","يطالب","رد","علق"]

# --- Liste des noms de déclarants
nom_declarant = [
    "هوكشتين","فرحان","عراقجي","سموتريتش","سعيد","كريم","قاسم","الشيخ","اورتاغس","رعد","الراعي","حاكم","لودريان",
    "جنبلاط","مفتي","جعجع","فارس","النايب","ترامب","سلامه","بري","خريس","الخازن","وديع","حماده","الموسوي","براك",
    "ارسلان","هاشم","بوتين","سلام","عون"
]

# --- Features linguistiques basées sur les verbes déclaratifs
df["contains_declarative_verb"] = df["titre"].apply(
    lambda t: int(any(v in str(t) for v in declarative_verbs))
)
df["starts_with_verb"] = df["titre"].apply(
    lambda t: int(any(str(t).strip().startswith(v) for v in declarative_verbs))
)

# --- Nouvelle feature : présence du nom d'un déclarant
df["contains_nom_declarant"] = df["titre"].apply(
    lambda t: int(any(n in str(t) for n in nom_declarant))
)

# --- Affichage lisible des 4 premières lignes
print("=== Aperçu des 4 premières lignes avec les nouvelles features ===")
display(
    df.head(4).style.set_properties(
        **{
            'text-align': 'right',      # Alignement à droite pour l'arabe
            'direction': 'rtl',         # Lecture RTL pour l'arabe
            'font-size': '16px',        # Texte plus grand
            'font-family': 'Arial'      # Police claire
        }
    )
)



=== Aperçu des 4 premières lignes avec les nouvelles features ===


,titre,label,contains_declarative_verb,starts_with_verb,contains_nom_declarant
0,"['عون:', 'لحكومه', 'تمثل', 'الجميع', 'مطالب', 'كتل', 'توخر', 'التاليف']",yes,0,0,1
1,"['زياره', 'ايرانيه', 'للراعي:', 'رساله', 'وتطمينات', 'وسجاده', 'بقطب', 'مخفيه']",no,0,0,0
2,"['تفاول', 'بحل', 'قضيه', 'الودايع:', 'اسس']",no,0,0,0
3,"['بوادر', 'حمله', 'عسكريه', 'اسراييليه', 'غربيه:', 'صنعاء', 'تتحضر', 'لتصعيد', 'وشيك']",no,0,0,0


In [7]:

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from IPython.display import display

# --- TF-IDF sur les titres
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_tfidf = vectorizer.fit_transform(df["titre"])

#  Inclure maintenant les 3 features linguistiques
X_extra = csr_matrix(df[["contains_declarative_verb", "starts_with_verb", "contains_nom_declarant"]].values)

# Combiner TF-IDF + features linguistiques
X = hstack([X_tfidf, X_extra])

# Labels
y = df["label"].values


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# --- Division des données
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Entraînement du modèle
model = LogisticRegression(max_iter=1000, solver="lbfgs")
model.fit(X_train, y_train)

# --- Prédictions
y_pred = model.predict(X_test)
# --- Évaluation
print("=== Rapport de classification ===")
print(classification_report(y_test, y_pred, digits=3))

print(" Précision globale :", round(accuracy_score(y_test, y_pred) * 100, 2), "%")



# --- Sauvegarder les résultats
joblib.dump(model, "declaration_classifier.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")


=== Rapport de classification ===
              precision    recall  f1-score   support

          no      0.636     0.726     0.678       106
         yes      0.716     0.624     0.667       117

    accuracy                          0.673       223
   macro avg      0.676     0.675     0.673       223
weighted avg      0.678     0.673     0.672       223

 Précision globale : 67.26 %


['tfidf_vectorizer.joblib']

In [10]:
import joblib
from google.colab import files

# --- Création d'un DataFrame avec les résultats
df_results = pd.DataFrame({
    "Titre": df["titre"].iloc[y_test.index if hasattr(y_test, 'index') else range(len(y_test))],
    "Label Réel": y_test,
    "Prédiction": y_pred,
    "contains_declarative_verb": df["contains_declarative_verb"].iloc[
        y_test.index if hasattr(y_test, 'index') else range(len(y_test))
    ].values,
    "starts_with_verb": df["starts_with_verb"].iloc[
        y_test.index if hasattr(y_test, 'index') else range(len(y_test))
    ].values,
    "contains_nom_declarant": df["contains_nom_declarant"].iloc[
        y_test.index if hasattr(y_test, 'index') else range(len(y_test))
    ].values
})

# --- Sauvegarder les prédictions dans Excel
df_results.to_excel("predictions_with_features.xlsx", index=False)

# --- Sauvegarder le modèle et le TF-IDF vectorizer
joblib.dump(model, "declaration_classifier.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")

# --- Télécharger les fichiers
files.download("predictions_with_features.xlsx")
files.download("declaration_classifier.joblib")
files.download("tfidf_vectorizer.joblib")
from IPython.display import display

# --- Afficher les 5 premières lignes du DataFrame des résultats
print("=== Aperçu des 5 premières lignes ===")
display(
    df_results.head(5).style.set_properties(
        **{
            'text-align': 'right',   # Alignement à droite pour l'arabe
            'direction': 'rtl',      # Lecture de droite à gauche
            'font-size': '16px',     # Taille du texte plus grande
            'font-family': 'Arial'   # Police claire pour l'arabe
        }
    )
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

=== Aperçu des 5 premières lignes ===


,Titre,Label Réel,Prédiction,contains_declarative_verb,starts_with_verb,contains_nom_declarant
0,"['عون:', 'لحكومه', 'تمثل', 'الجميع', 'مطالب', 'كتل', 'توخر', 'التاليف']",no,yes,0,0,1
1,"['زياره', 'ايرانيه', 'للراعي:', 'رساله', 'وتطمينات', 'وسجاده', 'بقطب', 'مخفيه']",yes,yes,0,0,0
2,"['تفاول', 'بحل', 'قضيه', 'الودايع:', 'اسس']",yes,no,0,0,0
3,"['بوادر', 'حمله', 'عسكريه', 'اسراييليه', 'غربيه:', 'صنعاء', 'تتحضر', 'لتصعيد', 'وشيك']",no,no,0,0,0
4,"['وزير', 'الخارجيه', 'السعودي', 'بيروت', 'الخميسين', 'وتشاوم', 'مسيحي', 'صعود', 'الدخان', 'الابيض', 'الرياسي']",no,yes,0,0,0
